# Notebook 07: Visualización Analítica y Gráficos Interactivos (Plotly)
**Proyecto**: AgroStats Intelligence Platform (`AgroStatsApp`)  
**Fase PDCO**: DEVELOPMENT $\rightarrow$ CONTROL | **Capa**: Gold $\rightarrow$ Analytics & Reporting  
**Estándar**: Tufte Principles / SWEBOK v4 / ISO/IEC 25010  

---
### Objetivos de la Suite Visual
1. **Monitoreo Hidrometeorológico**: Líneas interactivas de SPI-3 con umbrales normativos de sequía moderada/severa.
2. **Termometría Climatológica**: Barras divergentes de anomalías térmicas ($Z_T$) identificando olas de calor y descensos térmicos anómalos.
3. **Economía Agraria e Insumos**: Gráficos de eje dual relacionando precios nominales de fertilizantes con su inflación interanual (YoY).
4. **Benchmarking de Productividad**: Diagramas de barras ordenados para comparar rendimientos departamentales (Ton/Ha).

In [1]:
# Celda 1: Setup e Importaciones de Módulos Visuales
import sys
from pathlib import Path
import numpy as np
import pandas as pd

PROJECT_ROOT = Path("..").resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.config.settings import settings
from src.utils.logger import get_logger
from src.indicators.indicator_store import IndicatorStore
from src.visualization.plots import (
    plot_spi_timeline,
    plot_thermal_anomalies,
    plot_price_inflation,
    plot_yield_by_department,
)

logger = get_logger("notebooks.07_visualization")
store = IndicatorStore()
print("Módulos de visualización Plotly cargados exitosamente.")

2026-10-08T10:27:08-0500 | INFO     | agrostats.indicators.climate_indices:<module>:22 | Scipy no detectado; utilizando motor estadístico analítico puro para SPI


Módulos de visualización Plotly cargados exitosamente.


### 1. Monitoreo Hidrológico: Serie Temporal del Índice de Precipitación Estandarizado (SPI-3)

In [2]:
# Celda 2: Carga de serie de indicadores Gold y gráfico SPI
try:
    df_gold = store.load_indicator_dataset("indicadores_agroclimaticos_gold")
except Exception:
    # Generación de serie ilustrativa en caso de ejecución aislada
    fechas = pd.date_range("2021-01-01", periods=36, freq="ME")
    df_gold = pd.DataFrame({
        "fecha": fechas,
        "spi_3": np.random.normal(0, 1, 36),
        "anomalia_termica_z": np.random.normal(0, 1.2, 36),
        "precio_urea_bulto": 100000 + np.cumsum(np.random.normal(1000, 3000, 36)),
        "inflacion_yoy_urea": np.random.normal(12.0, 5.0, 36),
    })

fig_spi = plot_spi_timeline(df_gold, date_col="fecha", spi_col="spi_3")
fig_spi.show()

ValueError: Mime type rendering requires nbformat>=4.2.0 but it is not installed

### 2. Termometría Climatológica: Barras Divergentes de Anomalías Térmicas ($Z_T$)

In [3]:
# Celda 3: Gráfico divergente de anomalías térmicas
fig_temp = plot_thermal_anomalies(df_gold, date_col="fecha", anomaly_col="anomalia_termica_z")
fig_temp.show()

### 3. Dinámica Económica: Eje Dual para Inflación Interanual (YoY) de Fertilizantes

In [4]:
# Celda 4: Gráfico con eje dual de precio e inflación de insumos
fig_market = plot_price_inflation(
    df_gold,
    date_col="fecha",
    price_col="precio_urea_bulto",
    inflation_col="inflacion_yoy_urea",
)
fig_market.show()

### 4. Productividad Agrícola: Ranking Departamental de Rendimientos (Ton/Ha)

In [5]:
# Celda 5: Comparativa departamental de rendimientos de Papa
df_yield_sample = pd.DataFrame([
    {"departamento": "CUNDINAMARCA", "rendimiento_ton_ha": 22.4},
    {"departamento": "BOYACA", "rendimiento_ton_ha": 20.8},
    {"departamento": "NARINO", "rendimiento_ton_ha": 19.5},
    {"departamento": "ANTIOQUIA", "rendimiento_ton_ha": 17.2},
    {"departamento": "TOLIMA", "rendimiento_ton_ha": 14.6},
    {"departamento": "SANTANDER", "rendimiento_ton_ha": 13.8},
])

fig_yield = plot_yield_by_department(df_yield_sample, crop_name="Papa")
fig_yield.show()

### 5. Exportación de Tablero HTML Interactivo Autónomo

In [6]:
# Celda 6: Exportación de figuras a HTML para consumo web sin servidor
reports_dir = settings.BASE_DIR / "reports"
reports_dir.mkdir(parents=True, exist_ok=True)
dashboard_path = reports_dir / "dashboard_interactivo.html"

with open(dashboard_path, "w", encoding="utf-8") as f:
    f.write("<html><head><title>AgroStats Intelligence Dashboard</title></head><body style='font-family: Arial, sans-serif; padding: 20px;'>")
    f.write("<h1>AgroStats Intelligence Platform - Panel Ejecutivo</h1>")
    f.write(fig_spi.to_html(full_html=False, include_plotlyjs='cdn'))
    f.write("<hr style='margin: 40px 0;'>")
    f.write(fig_market.to_html(full_html=False, include_plotlyjs=False))
    f.write("</body></html>")

print(f"Dashboard exportado exitosamente: {dashboard_path}")